In [1]:
###### OSVAS ########################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)###########################
###### STEP 2: Downloading validation data from KMNI #############
#### STEP 2.0: DEFINING STATION and OSVAS PATH ########################
import os
# Default values defined in the notebook for Station and OSVAS install:
OSVAS='/home/pn56/OSVASgh/'  # Main OSVAS path
Station_name='Cabauw'

# If an environment variable STATION or OSVAS exists, override the default
Station_name = os.getenv("STATION_NAME", Station_name)
OSVAS = os.getenv("OSVAS", OSVAS)

print(f"Creating Validation files for: {Station_name} with OSVAS installation in {OSVAS}" )

Creating Validation files for: Cabauw with OSVAS installation in /home/alvaro/master/TFM/OSVAS


In [2]:
###### OSVAS ########################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)###########################
#### STEP 2.1: IMPORTING NEEDED PACKAGES AND DEFINING FUNCTIONS ###############
from datetime import date, datetime, timedelta, timezone
from dateutil.relativedelta import relativedelta
from functools import reduce
import matplotlib
import matplotlib.pyplot as plt
from netCDF4 import Dataset, date2num
import numpy as np
import os
import pandas as pd
import pytz
import re
import requests
import sqlite3
import tempfile
import xarray as xr
import xml.etree.ElementTree as ET
import yaml

##############################################################################
##Here comes a series of functions for handling the forcing creation easily ##
##############################################################################

def apply_transformation(variable, op, val):
    if op == "+":
        return variable + float(val)
    elif op == "-":
        return variable - float(val)
    elif op == "*":
        return variable * float(val)
    elif op == "/":
        return variable / float(val)
    else:
        return variable  # No op

def datespan(startDate, endDate, delta=timedelta(days=1)):
    currentDate = startDate
    while currentDate < endDate:
        yield currentDate
        currentDate += delta

def download_file(endpoint, headers, filename, download_directory='./.nc_data'):
    """
        Downloads the file filename from the endpoint to download_directory. If open is set to true, 
        the function returns the opened Netcdf Dataset.

        Parameters:
            endpoint: The url of the endpoint to receive the list of files
            headers: Dictionary with request headers, Authentication is mandatory with the corresponding token.
            download_directory: The path to the directory where files will be downloaded.
    """
    result = requests.get('/'.join((endpoint, filename, 'url')), headers=headers)
    r = requests.get(result.json()['temporaryDownloadUrl'])
    r.raise_for_status()
    with tempfile.NamedTemporaryFile(suffix=".nc") as f:
        f.write(r.content)
        f.flush()
        ds = xr.open_dataset(f.name)  # or "h5netcdf"
    return ds

def fetch_file_list(endpoint, headers, payload):
    """
        File list returned by endpoint satisfying payload criteria.
        
        Parameters:
            endpoint: The url of the endpoint to receive the list of files
            headers: Dictionary with request headers, Authentication is mandatory with the corresponding token
            payload: Dictionary with optional parameters for the get request.
        
        return: A list with the filenames that satisfy the given criteria. This filenames can then be retrieved calling another KMNI API endpoint
        
    """
    file_list = []
    truncated = True
    while truncated:
        result = requests.get(endpoint, headers=headers, params=payload)
        result.raise_for_status()
        result = result.json()
        file_list += [i['filename'] for i in result['files']]
        truncated = result['isTruncated']
        if truncated:
            payload += {'nextPageToken': result['nextPageToken']}
    print(f"{len(file_list)} files will be processed")
    return file_list

def initialize_forcing_dataset(start_date, end_date, timedelta, station_data):
    time = pd.date_range(start=start_date, end=end_date, freq=f"{timedelta}min").tz_convert("UTC").tz_localize(None)
    forcing_dataset = xr.Dataset(
        coords = {
            "time": ("time", time,),},
        data_vars = {
            "FRC_TIME_STP": (("Number_of_points",), [common_timedelta*60.], {"units": "s", "description": "forcing time step"}),
            "LAT": (("Number_of_points",), [float(station_data['lat'])], {"description": "latitudes" , "units": "degrees"}),
            "LON": (("Number_of_points",), [float(station_data['lon'])], {"description": "longitudes", "units": "degrees"}),
            "ZS": (("Number_of_points",), [float(station_data['elev'])], {"description": "surface orography", "units": "m"}),
            "UREF": (("Number_of_points",), [float(station_data['height_V'])], {"description": "Reference_Height_for_Wind", "units": "m"}),
            "ZREF": (("Number_of_points",), [float(station_data['height_T'])], {"description": "Reference_Height", "units": "m"}),
            "Tair": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "air temperature", "units": "K", "ascii_name": "Forc_TA"}),
            "Qair": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "air specific humidity", "units": "Kg/Kg", "ascii_name": "Forc_QA"}),
            "Wind": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "wind speed", "units": "m/s", "ascii_name": "Forc_WIND"}),
            "DIR_SWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward direct shortwave radiation", "units": "W/m2", "ascii_name": "Forc_DIR_SW"}),
            "SCA_SWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward diffuse shortwave radiation", "units": "W/m2", "ascii_name": "Forc_SCA_SW"}),
            "LWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward longwave radiation", "units": "W/m2", "ascii_name": "Forc_LW"}),
            "PSurf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "surface pressure", "units": "Pa", "ascii_name": "Forc_PS"}),
            "Rainf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "rainfall rate", "units": "Kg/m2/s", "ascii_name": "Forc_RAIN"}),
            "Snowf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "snowfall rate", "units": "Kg/m2/s", "ascii_name": "Forc_SNOW"}),
            "CO2air": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "CO2 concentration", "units": "Kg/m3", "ascii_name": "Forc_CO2"}),
            "Wind_DIR": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "wind direction", "units": "deg", "ascii_name": "Forc_DIR"}),
    })
    forcing_dataset.time.encoding.update({
        'units': "seconds since 2014-01-01 00:00:00", 
        'calendar': 'gregorian',
        "dtype": "float64"})
    return forcing_dataset

def parse_variable_entry(entry_str, timedelta_minutes):
    if entry_str is None or entry_str.strip() == "":
        return None, "const", 0.0

    parts = [p.strip() for p in entry_str.split(",", maxsplit=1)]

    # Case: constant only
    if len(parts) == 1:
        val = parts[0]
        if val in ["-", "None", ""]:
            return None, "const", 0.0
        try:
            return None, "const", float(val)
        except ValueError:
            return val, None, None  # just a direct mapping

    # Case: transformation
    src, transform = parts
    if src in ["-", "None", ""]:
        try:
            return None, "const", float(transform)
        except ValueError:
            raise ValueError(f"Invalid constant value in entry: {entry_str}")

    if "timedelta" in transform:
        transform = transform.replace("timedelta", str(timedelta_minutes))

    op = transform[0]
    expr = transform[1:].strip()

    try:
        val = eval(expr, {}, {})  # safe eval of math expression
    except Exception as e:
        raise ValueError(f"Failed to evaluate expression '{expr}' in entry '{entry_str}': {e}")

    return src, op, val

def process_data(df, variable_map, station_info, start, end):
    df['valid_dttm'] = pd.to_datetime(df['TIMESTAMP'], utc=True)
    df = df[(df['valid_dttm'] >= start) & (df['valid_dttm'] <= end)].copy()

    # Drop rows with missing required vars
    source_vars = list(variable_map.values())
    df = df.dropna(subset=source_vars)

    # Add station metadata
    df["SID"] = int(station_info["SID"])
    df["SID"] = df["SID"].astype("Int64")  # optional if you want pandas nullable integer type
    df['lat'] = station_info['lat']
    df['lon'] = station_info['lon']
    df['elev'] = station_info['elev']

    # Rename variables
    df = df.rename(columns={v: k for k, v in variable_map.items()})
    selected_columns = ['valid_dttm', 'SID', 'lat', 'lon', 'elev'] + list(variable_map.keys())

    return df[selected_columns]

def retrieve_data(endpoint, start_date, end_date, key):
    """
    Generates a dataset with the data collected from endpoint between start_date and end_date. It requires the api_key parameter 
    to make the requests to the service.
    """
    dataset_name = endpoint.split("/datasets/")[1].split("/")[0]
    dataset_version = endpoint.split("/versions/")[1].split("/")[0]
    start_file = (f"{dataset_name}_{dataset_version}_{start_date:%Y%m}.nc")
    end_file = (f"{dataset_name}_{dataset_version}_{end_date:%Y%m}.nc")
    months = pd.date_range(start=start_date, end=end_date, freq="MS")
    filenames = [f"{dataset_name}_{dataset_version}_{date:%Y%m}.nc" for date in months]
    headers={"Authorization": api_key}
    payload = {
        'maxKeys': '1000',
        'sorting': 'asc',
        'orderBy': 'filename',
        'begin': start_file,
        'end': end_file
    }
    file_list = fetch_file_list(endpoint, headers, payload)
    files = []
    for file in filenames:
        if file in file_list:
            files.append(download_file(endpoint, headers, file).drop_vars('valid_dates'))
        else:
            print(f"⚠️ {file} not found in KMNI service, data for this month will be filled with nan")
    ds_values = xr.concat(files, dim='time', data_vars='minimal').sortby('time')
    ds_values = ds_values.sel(time=slice(start_date.tz_localize(None), end_date.tz_localize(None)))
    #ds_values['SWD'] = ds_values['SWD'].clip(min=0)
    #ds_values['LWD'] = ds_values['SWD'].clip(min=0)
    return ds_values

def upsample_to_common_timedelta(datasets, dfs, common_td):
    dfs_resampled = []

    for name, df in zip(datasets.keys(), dfs):
        orig_td = pd.to_timedelta(datasets[name]["timedelta"], unit="m")
        if orig_td == common_td:
            dfs_resampled.append(df)
        else:
            df = df.set_index("valid_dttm")
            df = df.resample(common_td).interpolate(method="linear")
            df = df.reset_index()
            dfs_resampled.append(df)

    return dfs_resampled

def write_params_config(forcing_path, forcing_dataset):
    ddtt = pd.Timestamp(forcing_dataset['time'].values[0])
    
    param_lines = [
        len(forcing_dataset['LAT']),
        len(forcing_dataset['time']),
        forcing_dataset['FRC_TIME_STP'].values[0],
        ddtt.year,
        ddtt.month,
        ddtt.day,
        ddtt.hour*3600,
        "\t".join(map(str, forcing_dataset['LON'].values)),
        "\t".join(map(str, forcing_dataset['LAT'].values)),
        "\t".join(map(str, forcing_dataset['ZS'].values)),
        "\t".join(map(str, forcing_dataset['ZREF'].values)),
        "\t".join(map(str, forcing_dataset['UREF'].values))
    ]
    with open(os.path.join(forcing_path, "Params_config.txt"), 'w') as f:
                f.write("\n".join(map(str, param_lines)) + "\n")

def write_forcing_ascii(forcing_path, forcing_dataset):
    for var_name, var in forcing_dataset.data_vars.items():
        if "ascii_name" in var.attrs:
            file = var.attrs['ascii_name']
            np.savetxt(f"{os.path.join(forcing_path, file)}.txt", var.values, delimiter='\t', fmt='%.6f')

In [3]:
os.chdir(OSVAS)
CONFIG_PATH = f"config_files/Stations/{Station_name}.yml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

station_info = config["Station_metadata"]
validation_data = config["Validation_data"]
common_obstable = validation_data.get("common_obstable", False)

start_date = pd.to_datetime(validation_data["validation_start"], utc=True)
end_date = pd.to_datetime(validation_data["validation_end"], utc=True)

closure_type = config.get("Station_metadata", {}).get("closure_type", 1) #Default value is 1.

datasets = {k: v for k, v in validation_data.items() if k.startswith("dataset") or k.startswith("dataset_")}

key_path = os.path.join(OSVAS,"KMNI_token.txt") #new file with KMNI credential (maybe modify cookie_ICOS to store all the keys for diferent services?)
api_key = open(key_path, "r").readline().strip()
endpoint = validation_data['dataset1']['doi']
get_file_response = requests.get(endpoint, headers={"Authorization": api_key})
get_file_response.raise_for_status()

In [4]:
###### OSVAS ###################################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)######################################
#### STEP 2.3: Main loop over datasets, abort if no data in range ##############
datasets = {k: v for k, v in validation_data.items() if k.startswith("dataset") or k.startswith("dataset_")}
timedeltas = [ds_info['timedelta'] for ds_name, ds_info in datasets.items()]
units_map = {}
dfs = []
for ds_name, ds_info in datasets.items():
    print(f"Processing {ds_name} from DOI: {ds_info['doi']}")
    doi = ds_info["doi"]
    variable_map_raw = ds_info["variables"]
    
    variable_map = {k: v for k, v in ds_info["variables"].items() if v is not None}
    units_map.update({k: v for k, v in ds_info["units"].items() if v is not None})
    validation_src = retrieve_data(doi, start_date, end_date, api_key)
    validation_df = validation_src.ffill(dim='time').bfill(dim='time').drop_dims('nv').to_pandas()
    validation_df.index = validation_df.index.tz_localize("UTC").rename("TIMESTAMP")
    validation_df = validation_df.reset_index()
    df_processed = process_data(validation_df, variable_map, station_info, start_date, end_date)
    if df_processed.empty:
        raise RuntimeError(
            f"❌ No data found in the time window ({start_date} to {end_date}) "
            f"for dataset {ds_name} (DOI: {doi}). Aborting."
        )

    dfs.append(df_processed)

Processing dataset1 from DOI: https://api.dataplatform.knmi.nl/open-data/v1/datasets/cesar_surface_radiation_lc1_t10/versions/v1.0/files
36 files will be processed


Processing dataset2 from DOI: https://api.dataplatform.knmi.nl/open-data/v1/datasets/cesar_surface_flux_lc1_t10/versions/v1.0/files


36 files will be processed


Processing dataset3 from DOI: https://api.dataplatform.knmi.nl/open-data/v1/datasets/cesar_soil_heat_lb1_t10/versions/v1.0/files


36 files will be processed


In [5]:
###### OSVAS ############################################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)###############################################
#### STEP 2.4: Harmonize resolution, merge datasets, apply SEB closure if needed ########
common_td = pd.to_timedelta(min(timedeltas), unit="m")  # Choose finest resolution
dfs_resampled = upsample_to_common_timedelta(datasets, dfs, common_td)
# Cell 6: Merge all datasets, produce Hcor and LEcor based in a closure method if necessary and all SEB components available.
df_merged = reduce(lambda left, right: pd.merge(left, right, on=['valid_dttm', 'SID', 'lat', 'lon', 'elev'], how='outer'), dfs_resampled)
df_merged = df_merged.sort_values("valid_dttm").reset_index(drop=True)
#check closure

In [6]:
###### OSVAS ############################################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)###############################################
#### STEP 2.5: Convert to Unix timestamp in seconds and save dataframe to SQLite#########

import sqlite3
import os
import pandas as pd

# --- 1️⃣ Preprocess dataframe ---
df_merged["valid_dttm"] = pd.to_datetime(df_merged["valid_dttm"], utc=True)
df_merged["year_obs"] =df_merged["valid_dttm"].dt.year  # extract year for splitting
df_merged["valid_dttm"] = df_merged["valid_dttm"].apply(lambda x: int(x.timestamp()))
#df_merged["valid_dttm"] =_to_datetime_series(df_merged["valid_dttm"])

output_dir = (
    "sqlites/validation_data/common_obstables"
    if common_obstable
    else f"sqlites/validation_data/{station_info['Station_name']}"
)
os.makedirs(output_dir, exist_ok=True)

# --- 2️⃣ Loop over years ---
for year, df_year in df_merged.groupby("year_obs"):
    output_file = os.path.join(output_dir, f"OBSTABLE_{year}.sqlite")

    with sqlite3.connect(output_file) as conn:
        incoming_cols = list(df_year.columns)

        # --- 1️⃣ Build CREATE TABLE statement with correct types ---
        col_defs = []
        for c in incoming_cols:
            if c == "valid_dttm":
                col_defs.append(f'"{c}" INTEGER')
            elif c == "SID":
                col_defs.append(f'"{c}" DOUBLE')                
            else:
                col_defs.append(f'"{c}" REAL')

        conn.execute(f"""
            CREATE TABLE IF NOT EXISTS SYNOP (
                {", ".join(col_defs)},
                UNIQUE("valid_dttm","SID")
            );
        """)

        # --- 2️⃣ Detect existing columns ---
        existing_cols = [row[1] for row in conn.execute("PRAGMA table_info(SYNOP);")]

        # --- 3️⃣ Add new columns as REAL (except SID, which should already exist) ---
        for col in incoming_cols:
            if col not in existing_cols:
                if col in ("SID","valid_dttm"):
                    conn.execute(f'ALTER TABLE SYNOP ADD COLUMN "{col}" INTEGER;')
                else:
                    conn.execute(f'ALTER TABLE SYNOP ADD COLUMN "{col}" REAL;')

        # --- 4️⃣ Refresh column list ---
        existing_cols = [row[1] for row in conn.execute("PRAGMA table_info(SYNOP);")]

        # --- 5️⃣ Fill any missing columns in df ---
        for col in existing_cols:
            if col not in df_year.columns:
                df_year[col] = None

        # --- 6️⃣ Enforce integer type for SID before writing ---
        if "SID" in df_year.columns:
            df_year["SID"] = pd.to_numeric(df_year["SID"], errors="coerce").astype("Int64")

        df_year = df_year[existing_cols]

        # --- 7️⃣ Write to temporary table ---
        df_year.to_sql("SYNOP_tmp", conn, if_exists="replace", index=False)

        # --- 8️⃣ Merge logic ---
        conn.execute("""
            DELETE FROM SYNOP
            WHERE (valid_dttm, SID) IN (
                SELECT valid_dttm, SID FROM SYNOP_tmp
            );
        """)

        col_names = ", ".join([f'"{c}"' for c in existing_cols])
        conn.execute(f"""
            INSERT INTO SYNOP ({col_names})
            SELECT {col_names} FROM SYNOP_tmp;
        """)

        conn.execute("DROP TABLE SYNOP_tmp")
        # ---- create SYNOP_params if missing ----
        conn.execute("""
            CREATE TABLE IF NOT EXISTS SYNOP_params (
                parameter VARCHAR PRIMARY KEY,
                accum_hours REAL,
                units VARCHAR
            );
        """)

        # ---- get SYNOP columns from the actual table schema ----
        synop_cols = [row[1] for row in conn.execute("PRAGMA table_info(SYNOP);")]

        # ---- define which columns are metadata and should NOT be listed in SYNOP_params ----
        skip_cols = {"SID", "valid_dttm", "lat", "lon", "elev", "year_obs"}

        # ---- find which parameter names are already present to avoid duplicates ----
        existing_params = {row[0] for row in conn.execute("SELECT parameter FROM SYNOP_params;")}

        # ---- prepare rows for insertion: only column names in SYNOP that are not metadata and not already present ----
        rows_to_insert = []
        for col in synop_cols:
            if col in skip_cols:
                continue
            if col in existing_params:
                continue
            unit = units_map.get(col, "")   # default to empty string if not in units_map
            rows_to_insert.append((col, 0.0, unit))

        # ---- insert missing parameter rows ----
        if rows_to_insert:
            conn.executemany(
                "INSERT INTO SYNOP_params (parameter, accum_hours, units) VALUES (?, ?, ?);",
                rows_to_insert
            )
            conn.commit()        

    print(f"✅ Year {year} data merged into {output_file}")


✅ Year 2018 data merged into sqlites/validation_data/Cabauw/OBSTABLE_2018.sqlite


✅ Year 2019 data merged into sqlites/validation_data/Cabauw/OBSTABLE_2019.sqlite


✅ Year 2020 data merged into sqlites/validation_data/Cabauw/OBSTABLE_2020.sqlite
